# 03 — Mempool & On-Chain Metrics
## Collecte mempool.space + Glassnode

**Duree** : ~1-2h pour tout
**Sources** :
- mempool.space API (etat du mempool, fee rates)
- Glassnode free tier (SOPR, active addresses, exchange flows, hashrate)

v2: BGeometrics supprime (utilite < 3/10 pour horizons 30s-5min).

In [ ]:
import os, sys, time
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.collectors.mempool import (
    fetch_mempool_snapshot, run_mempool_collection,
)
from btc_pipeline.collectors.glassnode import (
    run_glassnode_collection,
)
# v2: removed — run_bgeometrics_collection (utilite < 3/10)

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()
config = Config()

## Section 1 — Snapshot mempool.space actuel

In [ ]:
# Test d'un snapshot mempool
snapshot = fetch_mempool_snapshot(config)
if snapshot:
    import pandas as pd
    print("Snapshot mempool actuel :")
    for k, v in snapshot.items():
        print(f"  {k}: {v}")
else:
    print("⚠️  mempool.space non accessible")

## Section 2 — Métriques Glassnode (free tier)

In [ ]:
# Collecte Glassnode
if config.glassnode_api_key:
    print("▶ Collecte Glassnode...")
    stats_gn = run_glassnode_collection(storage, config=config)
    print(f"✅ Glassnode : {stats_gn['metrics_collected']} métriques, {stats_gn['total_rows']:,} rows")
else:
    print("⚠️  Pas de clé API Glassnode (GLASSNODE_API_KEY)")
    print("   Les métriques Glassnode seront manquantes dans le dataset")
    print("   Obtenir une clé gratuite : https://studio.glassnode.com/")

## Section 3 — Collecte mempool continue (optionnel)
Lance la collecte toutes les 60 secondes. Laisser tourner en arriere-plan.

In [ ]:
# Collecte mempool continue (arrêter avec Ctrl+C)
# Durée par défaut : 24h. Modifier duration_hours selon besoin.
# Pour une collecte permanente, utiliser le notebook 06 (WebSocket).

COLLECT_HOURS = 1  # Mettre 24 pour une journée complète

print(f"▶ Collecte mempool pendant {COLLECT_HOURS}h (Ctrl+C pour arrêter)")
try:
    stats_mp = run_mempool_collection(
        storage,
        interval_seconds=60,
        duration_hours=COLLECT_HOURS,
        config=config,
    )
    print(f"✅ Mempool : {stats_mp['snapshots']} snapshots collectés")
except KeyboardInterrupt:
    print("⏹  Collecte mempool arrêtée")

In [ ]:
# Resume
print("\n" + "=" * 55)
print("RESUME — MEMPOOL & ON-CHAIN (v2)")
print("=" * 55)
# v2: removed bgeometrics prefix
for prefix in ["raw/mempool", "raw/onchain_metrics/glassnode"]:
    files = storage.list_files(prefix)
    print(f"  {prefix}: {len(files)} files")
print("=" * 55)